In [1]:
# Packages to Install for Scraping
%pip -q install requests beautifulsoup4 
import requests, json
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import hashlib
import os

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Fetch the HTML content
url = "https://www.boston.gov/public-notices/16600326"
response = requests.get(url)
#print(response.text)
soup = BeautifulSoup(response.text, 'html.parser')
#print(soup.prettify())


In [3]:
title=soup.title.string
detail_url = url
notice_id = 16600326


#Make the folder for the pdfs if it doesnt exist
folder_name = "public-notice-pdfs"
notice_folder = os.path.join(folder_name, str(notice_id))
os.makedirs(notice_folder, exist_ok=True)

# Finding the Posted Date
posted_label=soup.find("div",class_="dl-t", string=lambda t:t and "Posted" in t)
posted_raw=posted_label.find_next_sibling("div",class_="dl-d").get_text(strip=True)
posted_at = datetime.strptime(posted_raw, "%m/%d/%Y - %I:%M%p").replace(tzinfo=ZoneInfo("America/New_York")).isoformat()

# Discussion Topics Text
discussion_label = soup.find("h2",class_="header-border-bottom", string=lambda t:t and "Discussion Topics" in t)
discussion_text = discussion_label.find_next_sibling("div",class_="body").get_text(strip=False)


# Event details
event_date_container = soup.find("div", class_="date-title")
event_datetime = event_date_container.find("time")["datetime"]
address_container = soup.find("div",class_="detail-item__body--secondary sb-d")
address_line_1 = address_container.find("span",class_="address-line1").get_text(strip=False)
address_line_2 = address_container.find("span",class_="address-line2").get_text(strip=False)

#Look for public comment
public_testimony = False
testimony = soup.find("div",class_="n-li-a", string=lambda t:t and "The public can offer testimony" in t)
if testimony:
    public_testimony = True

# Look for cancellation
cancelled = False
cancellation = soup.find("span",class_="t--err t--s60pct", string=lambda t:t and "Canceled" in t)
if cancellation:
    cancelled=True
    
# PDFS
files = []
resources_label = soup.find("div", class_="sb-t", string=lambda t: t and "Resources" in t)
resources_container = resources_label.find_parent("div", class_="detail-item__content")
pdf_links = resources_container.select("div.link-wrapper.download-link a")

files = [{"file_label": a.get_text(strip=True), "file_url": a["href"]} for a in pdf_links]

#Hashing for Files
def hash_sha256(data:bytes):
    return hashlib.sha256(data).hexdigest()

# Download the files


for file in files:
    response = requests.get(file["file_url"])
    if response.status_code == 200: 
        file_path = os.path.join(folder_name,str(notice_id),file["file_label"])
        with open(file_path, 'wb') as f:
            f.write(response.content)
        file["download_success"] = True
        file["file_hash"] = hash_sha256(response.content)
    else:
        file["download_success"] = False
        file["file_hash"]= None 

    file["added_to_chroma"]=False



record = {
    "notice_id": notice_id,
    "title": title,
    "cancelled": cancelled,
    "detail_url": detail_url,
    "posted_at": posted_at,
    "event_datetime": event_datetime,
    "address_1":address_line_1,
    "address_2":address_line_2,
    "page_text": discussion_text,
    "files": files,
    "status": "ok",
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "page_text_added_to_chroma":False,
}
#print(record) 

PDFs to ChromaDB

In [4]:
%pip -q install pandas langchain langchain-core langchain-community langchain-chroma langchain-huggingface chromadb sentence-transformers transformers accelerate sentencepiece langchain-docling
import pandas as pd

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from docling.chunking import HybridChunker
from langchain_docling import DoclingLoader
from pathlib import Path
import shutil
import re
from langchain_docling.loader import ExportType
from langchain_text_splitters import RecursiveCharacterTextSplitter


Note: you may need to restart the kernel to use updated packages.


c:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHROMA_DB_PATH = "chroma_db"
COLLECTION_NAME = "public_notices"
EXPORT_TYPE = ExportType.DOC_CHUNKS

# MAKE THE CHROMADB
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma(
    collection_name = COLLECTION_NAME,
    embedding_function = embeddings,
    persist_directory=CHROMA_DB_PATH,
)

#Make metadata from Record
notice_metadata = {
    "notice_id": record["notice_id"],
    "title": record["title"],
    "cancelled": record["cancelled"],
    "detail_url": record["detail_url"],
    "posted_at": record["posted_at"],
    "event_datetime": record["event_datetime"],
    "address_1": record["address_1"],
    "address_2": record["address_2"],
    "status": record["status"],
    "checked_at": record["checked_at"],
}

#Add PDFs to Chroma

for file in files:
    # Skip anything already embedded, or that never downloaded successfully
    if file["added_to_chroma"] == True or file["download_success"] == False:
        continue

    file_path = os.path.join(notice_folder, file["file_label"])

    try:
        loader = DoclingLoader(
            file_path=file_path,
            export_type=EXPORT_TYPE,
            chunker=HybridChunker(tokenizer=EMBEDDING_MODEL)
        )
        docs = loader.load()

        # Add Metadata
        for doc in docs:
            # Chroma metadata values must be flat scalars (str, int, float, bool).
            # Docling attaches "dl_meta", a deeply nested dict, which makes
            # add_documents() raise and no PDF chunks get stored. So drop it here.
            doc.metadata.pop("dl_meta", None)
            # "source" is the local file path on whoever ran this, not useful downstream
            doc.metadata.pop("source", None)

            doc.metadata.update(notice_metadata)
            doc.metadata.update({
                "file_label": file["file_label"],
                "file_hash": file["file_hash"],
                "source_type": "pdf",
            })

        #Add to chroma
        vectorstore.add_documents(docs)
        file["added_to_chroma"] = True
        print(f"Added {len(docs)} chunks from '{file['file_label']}'")

    except Exception as e:
        # Printing reason
        print(f"FAILED to add PDF '{file['file_label']}': {e}")
        file["added_to_chroma"] = False

#Add page text to Chroma
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100
)

if not record.get("page_text_added_to_chroma") and record["page_text"].strip():
    try:
        page_docs = text_splitter.create_documents(
            texts=[record["page_text"]],
            metadatas=[{
                **notice_metadata,
                "source_type": "page_text",
            }],
        )

        vectorstore.add_documents(page_docs)
        record["page_text_added_to_chroma"] = True
        print(f"Added {len(page_docs)} chunks from page text")

    except Exception as e:
        # Printing reason
        print(f"FAILED to add page text: {e}")
        record["page_text_added_to_chroma"] = False

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3805.08it/s]


trying: public-notice-pdfs\16600326\Docket #1237 exists: True


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-07-29 00:44:32,061 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-29 00:44:32,081 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-29 00:44:32,081 [RapidOCR] main.py:63: Using C:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-29 00:44:32,221 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-29 00:44:32,225 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\shaik\Documents\Found

Docket #1237 -> 3 chunks
trying: public-notice-pdfs\16600326\Official Filed Posting exists: True


[INFO] 2026-07-29 00:44:38,236 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-29 00:44:38,260 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-29 00:44:38,264 [RapidOCR] main.py:63: Using C:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-29 00:44:38,379 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-29 00:44:38,380 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\shaik\Documents\Foundations of Gen AI\Project\CS6180_GenAI_Final_Project_City_of_Boston_RAG\venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-29 00:44:38,380 [RapidOCR] main.py:63: Using C:\Users\shaik\Documents\Fo

Official Filed Posting -> 4 chunks


In [6]:
print(vectorstore._collection.count())

12
